# External PDAC experimental-data harmonisation

**Purpose.** Reconstruct the experimental, pair-level response tables used to prepare the separate external benchmark in external_experimental_validation.ipynb. This notebook processes four published sources: Yale (Langdon et al., 2017), Höfer et al. (2025), Vis et al. (2024), and Bashi et al. (2024). The fifth source, NCATS/PANC-1 by Pourmousa et al., is processed in notebook external_experimental_validation.ipynb.

1. Download the files listed in the next table from the corresponding publication/repository. Respect each publisher's conditions and do not commit third-party raw data unless redistribution is permitted.
2. **Edit only the paths in the next code cell**. Either use repository-relative strings or full absolute paths (Windows paths work). Run the notebook within the repository, or set `PROJECT_ROOT_OVERRIDE` explicitly.
3. Run **Kernel → Restart & Run All**. The notebook intentionally refuses a missing input, a wrong Excel worksheet, and an unexpected schema or number of pairs rather than silently discovering an unrelated file.
4. Harmonised CSVs are saved under `results/tables/ml_comprehensive/` at the same locations used by the original thesis workflow. Outputs include file SHA-256 provenance, but do **not** include external model fitting or scoring.

### Exact original-source files and variables

| Study | Publication | **Exact downloaded item used in this notebook** | Selection / response |
|---|---|---|---|
| **Yale** — Langdon et al. (2017) | [DOI: 10.1158/1535-7163.MCT-16-0794](https://doi.org/10.1158/1535-7163.MCT-16-0794); data distributed through **SYNERGxDB** | `Raw_YALE-PDAC.zip` → internal `Pancreas_study_summary.csv`; `Pancreas_study_raw.csv` is **not needed** to produce the benchmark | `observed_bliss_diff`, also cross-checked against `combo_observed - bliss`. Remove controls and same-drug combinations; average per cell-line–pair and then per pair. **Continuous only**. |
| **Höfer** et al. (2025) | [DOI: 10.1038/s44320-025-00085-6](https://doi.org/10.1038/s44320-025-00085-6) | `hoefer_2025_Dataset_EV2.xlsx`, publisher Supplementary Dataset **EV2** (`44320_2025_85_MOESM4_ESM.xlsx`) | Worksheet `Dataset EV2_146Drugs_Synergy`; experimental `Synergy` flag and `AUC∆_sum [%]`. Retain 12 PDAC lines; **exclude HPDE** (non-malignant). Gemcitabine is the other drug in every pair. EV1 is **not** used. |
| **Vis** et al. (2024) | [DOI: 10.1016/j.xcrm.2024.101687](https://doi.org/10.1016/j.xcrm.2024.101687) | **Data S1 — Source data and code**, `mmc7.zip`, internal `data-code/SupplementalMaterials/pancan_combi_fitted_data_jan21.tsv.gz` | `ANCHOR_NAME`, `LIBRARY_NAME`, `CELL_LINE_NAME`, `tissue`, `SYNERGY_DELTA_EMAX`; only pancreatic rows, **continuous only**. Raw column name is retained for provenance; it is **not** the thesis's internal `ΔfAUC` and does not revive an obsolete candidate-response endpoint. |
| **Bashi** et al. (2024) | [DOI: 10.1158/2159-8290.CD-23-0388](https://doi.org/10.1158/2159-8290.CD-23-0388); [Figshare project 163378](https://figshare.com/projects/Large-scale_pan-cancer_cell_line_screening_identifies_actionable_and_effective_drug_combinations/163378) | **Supplementary Table S18**: `cd-23-0388_table_s18_suppst18.xlsx` | Worksheet `Supplementary Table 18 Single`. Actual `Compound1_name`, `Compound2_name`, `CANCER_TYPE` and `CELL_LINE_NAME`; response fields `combo_MaxE`, `HSA_matrix`, `Bliss_matrix`. The thesis pipeline **constructs** the binary criterion `(combo_MaxE > 0.5) AND (HSA_matrix > 0.1)` and uses `HSA_matrix` as the continuous outcome. Do not mistake the derived binary indicator for a separately supplied Boolean label. |

**Interpretation notes.** The four source responses are *not* a single harmonised pharmacological endpoint. Positive/negative classification is not invented for Yale or Vis. The same-pair/one-cold/both-cold novelty classes and Morgan-fingerprint coverage filtering are done **later, in notebook 41**, relative to the 26-drug internal Jaaks/GDSC training library. Source-publication links above locate the papers; the file and internal worksheet names come from the **executed supplied HTML**, not an assumption about generic supplementary downloads.

In [1]:
from pathlib import Path
import hashlib
import json
import re
import zipfile
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from IPython.display import display

# Otherwise specify the absolute path to the repository root, e.g.
PROJECT_ROOT_OVERRIDE = None

# These can be relative to PROJECT_ROOT or absolute paths. Edit individually.
YALE_ZIP_PATH = "data/raw/external_validation/yale_pdac/Raw_YALE-PDAC.zip"
HOEFER_EV2_PATH = "data/raw/external_validation/hoefer_2025/hoefer_2025_Dataset_EV2.xlsx"
VIS_TSV_PATH = "data/external/independent_validation/vis_2024_cell_reports_medicine/vis_2024_data_s1_source_data_and_code_extracted/data-code/SupplementalMaterials/pancan_combi_fitted_data_jan21.tsv.gz"
BASHI_S18_PATH = "data/raw/Bashi_2024/cd-23-0388_table_s18_suppst18.xlsx"

# The exact sheet names used in the completed thesis analysis.
HOEFER_SHEET = "Dataset EV2_146Drugs_Synergy"
BASHI_SHEET = "Supplementary Table 18 Single"

# All four inputs are necessary for the full harmonisation contract.
# Keep checks on for a strict thesis reproduction; set False for exploring
# a revised data release (in which case you cannot claim matching counts).
ENFORCE_THESIS_PAIR_COUNTS = True

# ------------------ do not change below this line ------------------
def identify_repo(start=None, override=None):
    if override is not None:
        p = Path(override).expanduser().resolve()
        if not (p / "notebooks").is_dir():
            raise FileNotFoundError(f"Invalid repository override (no notebooks/): {p}")
        return p
    start = (Path.cwd() if start is None else Path(start)).resolve()
    for parent in (start, *start.parents):
        if (parent / ".git").exists() and (parent / "notebooks").is_dir():
            return parent
    for parent in (start, *start.parents):
        if (parent / "notebooks").is_dir() and (parent / "data").is_dir():
            return parent
    raise FileNotFoundError("Cannot locate repository. Set PROJECT_ROOT_OVERRIDE in the configuration cell.")

PROJECT_ROOT = identify_repo(override=PROJECT_ROOT_OVERRIDE)
BASE = PROJECT_ROOT / "results" / "tables" / "ml_comprehensive"
OUT = BASE / "external_dataset_harmonisation"


def input_file(configured, description):
    p = Path(configured).expanduser()
    p = p.resolve() if p.is_absolute() else (PROJECT_ROOT / p).resolve()
    if not p.is_file():
        raise FileNotFoundError(
            f"Missing {description}: {p}\nEdit its path in the CONFIGURATION cell; "
            "the program does not search random folders or download files automatically."
        )
    return p


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def expect_count(actual, expected, label):
    if int(actual) != int(expected):
        message = f"{label}: expected {expected}, actually found {actual}"
        if ENFORCE_THESIS_PAIR_COUNTS:
            raise AssertionError(message)
        print("WARNING —", message)

pd.set_option("display.max_columns", 120)
print("Repository:", PROJECT_ROOT)
print("Outputs:", BASE)


Repository: C:\Users\Käyttäjä\Documents\PDAC-drug-combination-ML-MSc-thesis
Outputs: C:\Users\Käyttäjä\Documents\PDAC-drug-combination-ML-MSc-thesis\results\tables\ml_comprehensive


## A. Yale — Langdon 2017 / SYNERGxDB

Only the packaged **summary** CSV supplies external prediction outcomes. `Pancreas_study_raw.csv` contains underlying signal measurements but is not used in the pair-level scoring table. The notebook verifies the identity `observed_bliss_diff ≈ combo_observed - bliss`; its average is an empirical continuous response, **not** a new binary label.

In [2]:
YALE_ZIP = input_file(YALE_ZIP_PATH, "Yale Raw_YALE-PDAC.zip")
with zipfile.ZipFile(YALE_ZIP) as archive:
    # Match the exact filename, not any CSV whose name happens to contain 'summary'.
    summaries = [n for n in archive.namelist()
                 if Path(n).name.lower() == "pancreas_study_summary.csv"]
    if len(summaries) != 1:
        raise FileNotFoundError("Expected exactly one Pancreas_study_summary.csv in Yale zip; found: "
                                + repr(summaries))
    summary_raw = pd.read_csv(archive.open(summaries[0]))

print("Yale source:", YALE_ZIP)
print("Archive member:", summaries[0], "| raw summary shape:", summary_raw.shape)
display(summary_raw.head(3))


Yale source: C:\Users\Käyttäjä\Documents\PDAC-drug-combination-ML-MSc-thesis\data\raw\external_validation\yale_pdac\Raw_YALE-PDAC.zip
Archive member: Pancreas_study_summary.csv | raw summary shape: (45076, 15)


,expt_no,cell_line,inhibition_calc_method,first_agent,conc_first_agent,first_agent_inhibition,first_agent_stddev,second_agent,conc_second_agent,second_agent_inhibition,second_agent_stddev,combo_observed,combo_stddev,bliss,observed_bliss_diff
0,251,MiaPaCa-2,41,GSK650394A,2.1931,-22.35,8.046,Thapsigargin,0.0100,-17.57,9.3121,63.30,8.2290,-43.85,107.15
1,252,PANC-1,41,GSK2334470,0.1007,-4.48,13.216,Vorinostat,0.7687,-8.35,23.2130,93.65,4.6825,-13.20,106.85
2,251,MiaPaCa-2,41,GSK650394A,2.1931,-22.35,8.046,Vorinostat,0.4261,-4.78,4.2064,72.35,4.3410,-28.20,100.55


In [3]:
# ============================================================
# Basic schema and sanity checks
# ============================================================

required_cols = [
    "cell_line",
    "first_agent",
    "conc_first_agent",
    "second_agent",
    "conc_second_agent",
    "combo_observed",
    "bliss",
    "observed_bliss_diff",
]
missing = [c for c in required_cols if c not in summary_raw.columns]
if missing:
    raise ValueError(f"Missing required columns from Yale summary file: {missing}")

print("Columns:")
print(summary_raw.columns.tolist())

print("\nCell-line labels before filtering:")
display(summary_raw["cell_line"].value_counts(dropna=False).to_frame("n_rows"))

# Check that observed_bliss_diff equals combo_observed - bliss.
mask = summary_raw["observed_bliss_diff"].notna()
max_abs_diff = np.nanmax(
    np.abs((summary_raw.loc[mask, "combo_observed"] - summary_raw.loc[mask, "bliss"]) - summary_raw.loc[mask, "observed_bliss_diff"])
) if mask.any() else np.nan
print("Max absolute check: (combo_observed - bliss) - observed_bliss_diff =", max_abs_diff)

print("\nobserved_bliss_diff summary before filtering:")
display(summary_raw["observed_bliss_diff"].describe().to_frame().T)


Columns:
['expt_no', 'cell_line', 'inhibition_calc_method', 'first_agent', 'conc_first_agent', 'first_agent_inhibition', 'first_agent_stddev', 'second_agent', 'conc_second_agent', 'second_agent_inhibition', 'second_agent_stddev', 'combo_observed', 'combo_stddev', 'bliss', 'observed_bliss_diff']

Cell-line labels before filtering:


,n_rows
cell_line,
PANC-1,9140
PL45,9140
SU-86-86,7626
MiaPaCa-2,7058
HPAF-II,1514
AsPC-1,1514
Capan-2,1514
Capan-1,1514
Hs766T,1514


Max absolute check: (combo_observed - bliss) - observed_bliss_diff = 1.4210854715202004e-14

observed_bliss_diff summary before filtering:


,count,mean,std,min,25%,50%,75%,max
observed_bliss_diff,27930.0,7.317206,21.027326,-88.3,-4.3,4.68,18.69,107.15


In [4]:
# ============================================================
# Cleaning helpers
# ============================================================

CONTROL_TERMS = ["DMSO", "CONTROL", "VEHICLE"]
DROP_SELF_COMBINATIONS = True

# Important synonym map for matching against your training data and final candidates.
# This is intentionally conservative and editable.
DRUG_SYNONYM_MAP = {
    "MK-2206": "MK2206",
    "GSK1120212": "TRAMETINIB",
    "ABT-199": "VENETOCLAX",
    "BIBW-2992": "AFATINIB",
    "BYL-719": "ALPELISIB",
    "GDC-0941": "PICTILISIB",
    "5-FLUOROURACIL": "5-FLUOROURACIL",
}

CELL_LINE_SYNONYM_MAP = {
    "SU.86.86": "SU-86-86",
    "SU8686": "SU-86-86",
    "MIAPACA-2": "MiaPaCa-2",
    "MIA PACA-2": "MiaPaCa-2",
    "PANC1": "PANC-1",
}


def clean_token(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    x = x.replace("–", "-").replace("—", "-")
    x = re.sub(r"\s+", " ", x)
    return x


def clean_drug_name(x):
    x = clean_token(x)
    if pd.isna(x):
        return np.nan
    x = x.upper()
    return DRUG_SYNONYM_MAP.get(x, x)


def clean_cell_line_name(x):
    x0 = clean_token(x)
    if pd.isna(x0):
        return np.nan
    key = x0.upper().replace(" ", "")
    return CELL_LINE_SYNONYM_MAP.get(key, x0)


def is_control_name(x):
    if pd.isna(x):
        return True
    xu = str(x).upper()
    return any(term in xu for term in CONTROL_TERMS)


def make_pair_key(a, b):
    a = clean_drug_name(a)
    b = clean_drug_name(b)
    if pd.isna(a) or pd.isna(b):
        return np.nan
    return "__".join(sorted([a, b]))


def split_pair_key(pair_key):
    parts = str(pair_key).split("__")
    if len(parts) != 2:
        return pd.Series([np.nan, np.nan])
    return pd.Series(parts)

print("DROP_SELF_COMBINATIONS:", DROP_SELF_COMBINATIONS)
print("Drug synonym map:")
print(DRUG_SYNONYM_MAP)


DROP_SELF_COMBINATIONS: True
Drug synonym map:
{'MK-2206': 'MK2206', 'GSK1120212': 'TRAMETINIB', 'ABT-199': 'VENETOCLAX', 'BIBW-2992': 'AFATINIB', 'BYL-719': 'ALPELISIB', 'GDC-0941': 'PICTILISIB', '5-FLUOROURACIL': '5-FLUOROURACIL'}


In [5]:
# ============================================================
# Clean concentration-level Yale summary rows
# ============================================================

df = summary_raw.copy()

df["external_source"] = "YALE-PDAC_SYNERGxDB_raw_download"
df["external_cell_line_original"] = df["cell_line"].astype(str)
df["external_cell_line"] = df["cell_line"].map(clean_cell_line_name)

df["external_drug_a_original"] = df["first_agent"].astype(str)
df["external_drug_b_original"] = df["second_agent"].astype(str)
df["external_drug_a"] = df["first_agent"].map(clean_drug_name)
df["external_drug_b"] = df["second_agent"].map(clean_drug_name)
df["external_pair_key"] = df.apply(lambda r: make_pair_key(r["external_drug_a"], r["external_drug_b"]), axis=1)

# row flags
control_mask = df["first_agent"].map(is_control_name) | df["second_agent"].map(is_control_name)
missing_response_mask = df["observed_bliss_diff"].isna()
self_combo_mask = df["external_drug_a"] == df["external_drug_b"]

filter_report = {
    "initial_rows": int(len(df)),
    "missing_observed_bliss_diff_rows": int(missing_response_mask.sum()),
    "control_rows": int(control_mask.sum()),
    "self_combination_rows_before_response_filter": int(self_combo_mask.sum()),
}

clean = df.loc[~missing_response_mask & ~control_mask].copy()

if DROP_SELF_COMBINATIONS:
    self_after = clean["external_drug_a"].eq(clean["external_drug_b"])
    filter_report["self_combination_rows_after_response_filter_excluded"] = int(self_after.sum())
    clean = clean.loc[~self_after].copy()
else:
    filter_report["self_combination_rows_after_response_filter_excluded"] = 0

filter_report["clean_concentration_rows"] = int(len(clean))
filter_report["clean_cell_lines"] = int(clean["external_cell_line"].nunique())
filter_report["clean_unordered_pairs"] = int(clean["external_pair_key"].nunique())

print(json.dumps(filter_report, indent=2))

print("\nClean concentration-level cell-line counts:")
display(clean["external_cell_line"].value_counts().to_frame("n_concentration_rows"))

print("\nClean concentration-level response summary:")
display(clean["observed_bliss_diff"].describe().to_frame().T)

display(clean.head())


{
  "initial_rows": 45076,
  "missing_observed_bliss_diff_rows": 17146,
  "control_rows": 17146,
  "self_combination_rows_before_response_filter": 471,
  "self_combination_rows_after_response_filter_excluded": 471,
  "clean_concentration_rows": 27459,
  "clean_cell_lines": 4,
  "clean_unordered_pairs": 820
}

Clean concentration-level cell-line counts:


,n_concentration_rows
external_cell_line,
PANC-1,7380
PL45,7380
SU-86-86,7380
MiaPaCa-2,5319



Clean concentration-level response summary:


,count,mean,std,min,25%,50%,75%,max
observed_bliss_diff,27459.0,7.22857,20.919537,-88.3,-4.29,4.65,18.54,107.15


,expt_no,cell_line,inhibition_calc_method,first_agent,conc_first_agent,first_agent_inhibition,first_agent_stddev,second_agent,conc_second_agent,second_agent_inhibition,second_agent_stddev,combo_observed,combo_stddev,bliss,observed_bliss_diff,external_source,external_cell_line_original,external_cell_line,external_drug_a_original,external_drug_b_original,external_drug_a,external_drug_b,external_pair_key
0,251,MiaPaCa-2,41,GSK650394A,2.1931,-22.35,8.0460,Thapsigargin,0.0100,-17.57,9.3121,63.30,8.2290,-43.85,107.15,YALE-PDAC_SYNERGxDB_raw_download,MiaPaCa-2,MiaPaCa-2,GSK650394A,Thapsigargin,GSK650394A,THAPSIGARGIN,GSK650394A__THAPSIGARGIN
1,252,PANC-1,41,GSK2334470,0.1007,-4.48,13.2160,Vorinostat,0.7687,-8.35,23.2130,93.65,4.6825,-13.20,106.85,YALE-PDAC_SYNERGxDB_raw_download,PANC-1,PANC-1,GSK2334470,Vorinostat,GSK2334470,VORINOSTAT,GSK2334470__VORINOSTAT
2,251,MiaPaCa-2,41,GSK650394A,2.1931,-22.35,8.0460,Vorinostat,0.4261,-4.78,4.2064,72.35,4.3410,-28.20,100.55,YALE-PDAC_SYNERGxDB_raw_download,MiaPaCa-2,MiaPaCa-2,GSK650394A,Vorinostat,GSK650394A,VORINOSTAT,GSK650394A__VORINOSTAT
4,251,MiaPaCa-2,41,GSK650394A,2.1931,-22.35,8.0460,MK-2206,0.1602,-11.27,13.7494,58.57,4.0999,-36.14,94.71,YALE-PDAC_SYNERGxDB_raw_download,MiaPaCa-2,MiaPaCa-2,GSK650394A,MK-2206,GSK650394A,MK2206,GSK650394A__MK2206
5,260,PL45,41,MLN-8237,0.0162,-24.69,2.9628,Vorinostat,0.7687,-21.78,4.7916,42.01,4.2010,-51.85,93.86,YALE-PDAC_SYNERGxDB_raw_download,PL45,PL45,MLN-8237,Vorinostat,MLN-8237,VORINOSTAT,MLN-8237__VORINOSTAT


In [6]:
# ============================================================
# Final Yale unit- and pair-level continuous-response tables
# ============================================================

# Split the canonical unordered pair key into its two sorted drug names.
pair_parts = clean["external_pair_key"].apply(split_pair_key)
clean["external_pair_drug_a"] = pair_parts[0]
clean["external_pair_drug_b"] = pair_parts[1]

# First level: one equal-weight unit per cell line × unordered pair.
yale_unit = (
    clean
    .groupby(
        [
            "external_cell_line",
            "external_pair_key",
            "external_pair_drug_a",
            "external_pair_drug_b",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        n_concentration_rows=("observed_bliss_diff", "size"),
        mean_observed_bliss_diff=("observed_bliss_diff", "mean"),
        median_observed_bliss_diff=("observed_bliss_diff", "median"),
        max_observed_bliss_diff=("observed_bliss_diff", "max"),
        min_observed_bliss_diff=("observed_bliss_diff", "min"),
    )
)

yale_unit["external_unit_id"] = (
    yale_unit["external_cell_line"].astype(str)
    + "||"
    + yale_unit["external_pair_key"].astype(str)
)

# Second level: equal-weight the cell-line units to one row per unordered pair.
yale_pair = (
    yale_unit
    .groupby(
        [
            "external_pair_key",
            "external_pair_drug_a",
            "external_pair_drug_b",
        ],
        as_index=False,
    )
    .agg(
        n_external_units=("external_unit_id", "size"),
        n_external_cell_lines=("external_cell_line", "nunique"),
        mean_observed_bliss_diff_mean=("mean_observed_bliss_diff", "mean"),
        median_observed_bliss_diff_mean=("mean_observed_bliss_diff", "median"),
        max_observed_bliss_diff_mean=("mean_observed_bliss_diff", "max"),
        min_observed_bliss_diff_mean=("mean_observed_bliss_diff", "min"),
    )
    .rename(
        columns={
            "external_pair_key": "pair_key",
            "external_pair_drug_a": "drug_a",
            "external_pair_drug_b": "drug_b",
        }
    )
)

expect_count(yale_pair["pair_key"].nunique(), 820, "Yale pair count")

YALE_OUT = BASE / "external_validation_yale_pdac"
YALE_OUT.mkdir(parents=True, exist_ok=True)

clean_path = YALE_OUT / "yale_pdac_external_test_concentration_level_clean.csv"
unit_path = YALE_OUT / "yale_pdac_external_test_unit_level_continuous.csv"
yale_pair_path = YALE_OUT / "yale_pair_level_external_candidate_summary.csv"

clean.to_csv(clean_path, index=False)
yale_unit.to_csv(unit_path, index=False)
yale_pair.to_csv(yale_pair_path, index=False)

print("Saved Yale concentration table:", clean_path, clean.shape)
print("Saved Yale unit table:", unit_path, yale_unit.shape)
print("Saved Yale pair table:", yale_pair_path, yale_pair.shape)
display(yale_pair.head())


Saved Yale concentration table: C:\Users\Käyttäjä\Documents\PDAC-drug-combination-ML-MSc-thesis\results\tables\ml_comprehensive\external_validation_yale_pdac\yale_pdac_external_test_concentration_level_clean.csv (27459, 25)
Saved Yale unit table: C:\Users\Käyttäjä\Documents\PDAC-drug-combination-ML-MSc-thesis\results\tables\ml_comprehensive\external_validation_yale_pdac\yale_pdac_external_test_unit_level_continuous.csv (3168, 10)
Saved Yale pair table: C:\Users\Käyttäjä\Documents\PDAC-drug-combination-ML-MSc-thesis\results\tables\ml_comprehensive\external_validation_yale_pdac\yale_pair_level_external_candidate_summary.csv (820, 9)


,pair_key,drug_a,drug_b,n_external_units,n_external_cell_lines,mean_observed_bliss_diff_mean,median_observed_bliss_diff_mean,max_observed_bliss_diff_mean,min_observed_bliss_diff_mean
0,17-DMAG__5-FLUOROURACIL,17-DMAG,5-FLUOROURACIL,4,4,-5.706111,-5.095556,3.554444,-16.187778
1,17-DMAG__AFATINIB,17-DMAG,AFATINIB,4,4,-1.581389,-3.196667,7.631111,-7.563333
2,17-DMAG__ALPELISIB,17-DMAG,ALPELISIB,4,4,-4.615556,-5.037778,-0.012222,-8.374444
3,17-DMAG__AZD-1480,17-DMAG,AZD-1480,4,4,-5.858333,-3.757778,3.876667,-19.794444
4,17-DMAG__AZD-7762,17-DMAG,AZD-7762,4,4,-12.001944,-4.383889,-2.175556,-37.064444


## B. Höfer — 2025 Supplementary Dataset EV2

**EV2 only** (publisher file `44320_2025_85_MOESM4_ESM.xlsx`), worksheet `Dataset EV2_146Drugs_Synergy`. The synergy flag is a **source-supplied** label, not a threshold selected in this notebook. The original response file includes the non-malignant HPDE cell line; exclude it from the primary pancreatic-cancer pair summary. Each inhibitor is paired with gemcitabine.

Direct publisher supplementary file, for reference: [EV2 workbook](https://media.springernature.com/original/springer-static/esm/art%3A10.1038%2Fs44320-025-00085-6/MediaObjects/44320_2025_85_MOESM4_ESM.xlsx). Download manually and set `HOEFER_EV2_PATH` above.

In [7]:
EV2_PATH = input_file(HOEFER_EV2_PATH, "Höfer EV2 synergy workbook")
H_OUT = BASE / "external_hoefer_2025_cross_assay_validation"


def norm_text(x):
    if pd.isna(x):
        return ""
    return re.sub(r"[^a-z0-9]+", "", str(x).strip().lower())

# Confirm expected sheet explicitly before reading.
hoef_xl = pd.ExcelFile(EV2_PATH)
if HOEFER_SHEET not in hoef_xl.sheet_names:
    raise KeyError(f"Höfer sheet {HOEFER_SHEET!r} missing; available: {hoef_xl.sheet_names}")
print("Höfer source:", EV2_PATH)
print("Sheet:", HOEFER_SHEET)


Höfer source: C:\Users\Käyttäjä\Documents\PDAC-drug-combination-ML-MSc-thesis\data\raw\external_validation\hoefer_2025\hoefer_2025_Dataset_EV2.xlsx
Sheet: Dataset EV2_146Drugs_Synergy


In [8]:
# ============================================================
# Exact Höfer EV2 synergy parser
# ============================================================

SYNERGY_SHEET = HOEFER_SHEET

d = pd.read_excel(
    EV2_PATH,
    sheet_name=SYNERGY_SHEET,
    header=0,
)

print("Loaded sheet:", SYNERGY_SHEET)
print("Raw shape:", d.shape)

required_columns = [
    "Cell line",
    "Drug",
    "AUC∆_GEM low [%]",
    "AUC∆_GEM high [%]",
    "AUC∆_sum [%]",
    "AUC∆_sum_greater10",
    "any_comb_AUC_lower80",
    "any_comb_pEC50_greater6",
    "both_comb_R2_greater0.8",
    "Synergy",
    "Count",
]

missing = [c for c in required_columns if c not in d.columns]
if missing:
    raise KeyError(
        "The Höfer EV2 synergy sheet does not match the expected schema. "
        f"Missing columns: {missing}"
    )


def parse_excel_bool(series):
    def convert(x):
        if pd.isna(x):
            return np.nan
        if isinstance(x, (bool, np.bool_)):
            return bool(x)
        if isinstance(x, (int, float, np.integer, np.floating)) and x in (0, 1):
            return bool(x)
        s = str(x).strip().lower()
        if s in {"true", "1", "yes", "y", "t"}:
            return True
        if s in {"false", "0", "no", "n", "f"}:
            return False
        return np.nan
    return series.map(convert)


bool_columns = [
    "AUC∆_sum_greater10",
    "any_comb_AUC_lower80",
    "any_comb_pEC50_greater6",
    "both_comb_R2_greater0.8",
    "Synergy",
]
for c in bool_columns:
    d[c] = parse_excel_bool(d[c])


hoefer_long = pd.DataFrame({
    "cell_line": d["Cell line"].astype(str).str.strip(),
    "inhibitor": d["Drug"].astype(str).str.strip(),
    "auc_delta_gem_low": pd.to_numeric(d["AUC∆_GEM low [%]"], errors="coerce"),
    "auc_delta_gem_high": pd.to_numeric(d["AUC∆_GEM high [%]"], errors="coerce"),
    "sum_auc_delta": pd.to_numeric(d["AUC∆_sum [%]"], errors="coerce"),
    "auc_delta_sum_greater10": d["AUC∆_sum_greater10"],
    "any_comb_auc_lower80": d["any_comb_AUC_lower80"],
    "any_comb_pec50_greater6": d["any_comb_pEC50_greater6"],
    "both_comb_r2_greater0_8": d["both_comb_R2_greater0.8"],
    "published_synergy": d["Synergy"],
    "count": pd.to_numeric(d["Count"], errors="coerce"),
})

# Threshold-only sensitivity endpoint; preserve genuine missing response rows as NaN.
hoefer_long["sum_auc_delta_gt10_only"] = np.where(
    hoefer_long["sum_auc_delta"].notna(),
    (hoefer_long["sum_auc_delta"] > 10).astype(int),
    np.nan,
)

# HPDE is non-malignant and is excluded from the primary PDAC validation.
hoefer_long["is_primary_pdac"] = ~hoefer_long["cell_line"].map(norm_text).eq(norm_text("HPDE"))

# Integrity audit.
n_rows = len(hoefer_long)
n_drugs = hoefer_long["inhibitor"].nunique()
n_cell_lines = hoefer_long["cell_line"].nunique()
n_pdac_rows = int(hoefer_long["is_primary_pdac"].sum())
n_hpde_rows = int((~hoefer_long["is_primary_pdac"]).sum())
n_duplicate_drug_cell = int(hoefer_long.duplicated(["inhibitor", "cell_line"]).sum())
n_published_positive = int(hoefer_long["published_synergy"].eq(True).sum())
n_published_negative = int(hoefer_long["published_synergy"].eq(False).sum())
n_published_missing = int(hoefer_long["published_synergy"].isna().sum())

print("\n" + "=" * 96)
print("HÖFER EV2 SYNERGY TABLE QC")
print("=" * 96)
print(f"Rows:                    {n_rows:,}")
print(f"Unique inhibitors:       {n_drugs}")
print(f"Cell lines:              {n_cell_lines}")
print(f"Primary PDAC rows:       {n_pdac_rows:,}")
print(f"HPDE rows:               {n_hpde_rows:,}")
print(f"Duplicate drug×cell:     {n_duplicate_drug_cell}")
print(f"Published synergy TRUE:  {n_published_positive}")
print(f"Published synergy FALSE: {n_published_negative}")
print(f"Published synergy NA:    {n_published_missing}")
print("=" * 96)

expect_count(n_rows, 1898, "Höfer EV2 source rows")
expect_count(n_drugs, 146, "Höfer inhibitors")
expect_count(n_cell_lines, 13, "Höfer total cell lines")
assert n_duplicate_drug_cell == 0, "Unexpected duplicate inhibitor × cell-line rows."
expect_count(n_pdac_rows, 1752, "Höfer primary PDAC rows")
expect_count(n_hpde_rows, 146, "Höfer HPDE rows")

# Expected workbook-level published flag counts.
expect_count(n_published_positive, 51, "Höfer source positive labels")
expect_count(n_published_negative, 1845, "Höfer source negative labels")
expect_count(n_published_missing, 2, "Höfer missing published labels")

print("\nPrimary PDAC published-synergy counts:")
display(
    hoefer_long.loc[hoefer_long["is_primary_pdac"], "published_synergy"]
    .value_counts(dropna=False)
    .rename_axis("published_synergy")
    .reset_index(name="n_rows")
)

print("\nGenuine missing experimental rows:")
display(hoefer_long[hoefer_long["published_synergy"].isna()])

OUT.mkdir(parents=True, exist_ok=True)
hoefer_long.to_csv(OUT / "hoefer_ev2_synergy_long_parsed.csv", index=False)
print("Saved:", OUT / "hoefer_ev2_synergy_long_parsed.csv")


Loaded sheet: Dataset EV2_146Drugs_Synergy
Raw shape: (1898, 15)

HÖFER EV2 SYNERGY TABLE QC
Rows:                    1,898
Unique inhibitors:       146
Cell lines:              13
Primary PDAC rows:       1,752
HPDE rows:               146
Duplicate drug×cell:     0
Published synergy TRUE:  51
Published synergy FALSE: 1845
Published synergy NA:    2

Primary PDAC published-synergy counts:


,published_synergy,n_rows
0,False,1703
1,True,47
2,NaN,2



Genuine missing experimental rows:


,cell_line,inhibitor,auc_delta_gem_low,auc_delta_gem_high,sum_auc_delta,auc_delta_sum_greater10,any_comb_auc_lower80,any_comb_pec50_greater6,both_comb_r2_greater0_8,published_synergy,count,sum_auc_delta_gt10_only,is_primary_pdac
1512,Suit2-07,Roblitinib,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,True
1779,Colo-357,Umbralisib,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,True


Saved: C:\Users\Käyttäjä\Documents\PDAC-drug-combination-ML-MSc-thesis\results\tables\ml_comprehensive\external_dataset_harmonisation\hoefer_ev2_synergy_long_parsed.csv


In [9]:
# Equal-weighted pair-level Höfer outcome summary across malignant PDAC cell lines.
hp = hoefer_long.loc[hoefer_long["is_primary_pdac"]].copy()
hp["drug_a"]="GEMCITABINE"
hp["drug_b"]=hp["inhibitor"]
def _norm_drug(x): return re.sub(r"[^A-Z0-9]+","",str(x).upper())
def _pair(a,b): return "||".join(sorted([_norm_drug(a),_norm_drug(b)]))
hp["pair_key"]=[_pair(a,b) for a,b in zip(hp["drug_a"],hp["drug_b"])]
hoefer_pair=(
    hp.groupby(["pair_key","drug_a","drug_b","inhibitor"],as_index=False)
      .agg(
          n_external_rows=("pair_key","size"),
          n_external_cell_lines=("cell_line","nunique"),
          mean_sum_auc_delta=("sum_auc_delta","mean"),
          median_sum_auc_delta=("sum_auc_delta","median"),
          max_sum_auc_delta=("sum_auc_delta","max"),
          published_synergy_fraction=("published_synergy",lambda x: pd.Series(x).dropna().astype(bool).mean() if pd.Series(x).notna().any() else np.nan),
          n_published_synergy=("published_synergy",lambda x:int(pd.Series(x).eq(True).sum())),
          any_published_synergy=("published_synergy",lambda x:float(pd.Series(x).eq(True).any())),
      )
)
expect_count(hoefer_pair["pair_key"].nunique(), 146, "Höfer pair count")
H_OUT.mkdir(parents=True,exist_ok=True)
hoefer_pair_path=H_OUT/"hoefer_inhibitor_pair_level_summary.csv"
hoefer_pair.to_csv(hoefer_pair_path,index=False)
print("Saved Höfer pair-level table:",hoefer_pair_path,hoefer_pair.shape)


Saved Höfer pair-level table: C:\Users\Käyttäjä\Documents\PDAC-drug-combination-ML-MSc-thesis\results\tables\ml_comprehensive\external_hoefer_2025_cross_assay_validation\hoefer_inhibitor_pair_level_summary.csv (146, 12)


## C. Vis 2024 and Bashi 2024

**Vis:** use the **fitted-response TSV from Data S1**, *not* the classified response Tables S2/S3. Only `SYNERGY_DELTA_EMAX` supplies the continuous external outcome; the notebook does not assign a Vis binary outcome. The source's field name is recorded verbatim to make the provenance auditable.

**Bashi:** use **Table S18** and its `Supplementary Table 18 Single` sheet, *not* Supplementary Table S2 or anonymous drug-code mappings. The Boolean classification is a **derived analysis rule**, applied to continuous `combo_MaxE` and `HSA_matrix`; `HSA_matrix` is the continuous outcome retained for external correlation analysis. Missing values in these columns are reported before classification. The exact original Boolean rule is preserved for comparison with the historical thesis analysis; it should not be described as a distinct published label. 

In [10]:
VIS_PATH = input_file(VIS_TSV_PATH, "Vis 2024 Data S1 fitted-response TSV")
BASHI_PATH = input_file(BASHI_S18_PATH, "Bashi 2024 Supplementary Table S18 workbook")

vis_raw = pd.read_csv(VIS_PATH, sep="\t", compression="infer", low_memory=False)
xl = pd.ExcelFile(BASHI_PATH)
if BASHI_SHEET not in xl.sheet_names:
    raise KeyError(f"Bashi sheet {BASHI_SHEET!r} not found. Available sheets: {xl.sheet_names}")
bashi_raw = pd.read_excel(xl, sheet_name=BASHI_SHEET)

print("Vis source:", VIS_PATH, "| shape:", vis_raw.shape)
print("Bashi source:", BASHI_PATH, "| sheet:", BASHI_SHEET, "| shape:", bashi_raw.shape)


Vis source: C:\Users\Käyttäjä\Documents\PDAC-drug-combination-ML-MSc-thesis\data\external\independent_validation\vis_2024_cell_reports_medicine\vis_2024_data_s1_source_data_and_code_extracted\data-code\SupplementalMaterials\pancan_combi_fitted_data_jan21.tsv.gz | shape: (137738, 68)
Bashi source: C:\Users\Käyttäjä\Documents\PDAC-drug-combination-ML-MSc-thesis\data\raw\Bashi_2024\cd-23-0388_table_s18_suppst18.xlsx | sheet: Supplementary Table 18 Single | shape: (84055, 60)


In [11]:
def norm_col(x):
    return re.sub(r"[^a-z0-9]+", "", str(x).lower())


def norm_drug(x):
    if pd.isna(x):
        return ""
    return re.sub(r"[^A-Z0-9]+", "", str(x).upper())


ALIASES = {
    "5FU": "5FLUOROURACIL",
    "FLUOROURACIL": "5FLUOROURACIL",
    "5FLUOROURACIL": "5FLUOROURACIL",
    "MK1775": "ADAVOSERTIB",
    "AZD1775": "ADAVOSERTIB",
    "ADAVOSERTIB": "ADAVOSERTIB",
    "ABT263": "NAVITOCLAX",
    "NAVITOCLAX": "NAVITOCLAX",
    "ABT199": "VENETOCLAX",
    "MK2206": "MK2206",
    "SCH772984": "SCH772984",
    "OSI906": "LINSITINIB",
    "LINSITINIB": "LINSITINIB",
    "GSK1120212": "TRAMETINIB",
    "TRAMETINIB": "TRAMETINIB",
    "GDC0032": "TASELISIB",
    "TASELISIB": "TASELISIB",
    "GDC0068": "IPATASERTIB",
    "IPATASERTIB": "IPATASERTIB",
    "AZD6244": "SELUMETINIB",
    "SELUMETINIB": "SELUMETINIB",
    "AZD2281": "OLAPARIB",
    "OLAPARIB": "OLAPARIB",
    "AZD5363": "CAPIVASERTIB",
    "CAPIVASERTIB": "CAPIVASERTIB",
    "AZD6738": "CERALASERTIB",
    "CERALASERTIB": "CERALASERTIB",
    "PF02341066": "CRIZOTINIB",
    "BMS354825": "DASATINIB",
    "PD0332991": "PALBOCICLIB",
    "LY2157299": "GALUNISERTIB",
    "INCB018424": "RUXOLITINIB",
    "BAY439006": "SORAFENIB",
    "WNT974": "LGK974",
}


def canon_drug(x):
    k = norm_drug(x)
    return ALIASES.get(k, k)


def canon_pair(a, b):
    return "||".join(sorted([canon_drug(a), canon_drug(b)]))


In [12]:
#Vis source-specific response standardisation.

required_vis = [
    "ANCHOR_NAME",
    "LIBRARY_NAME",
    "CELL_LINE_NAME",
    "tissue",
    "SYNERGY_DELTA_EMAX",
]
missing_vis = [c for c in required_vis if c not in vis_raw.columns]
if missing_vis:
    raise KeyError(f"Vis source table is missing required columns: {missing_vis}")

vis = pd.DataFrame({
    "drug_a": vis_raw["ANCHOR_NAME"],
    "drug_b": vis_raw["LIBRARY_NAME"],
    "cell_line": vis_raw["CELL_LINE_NAME"],
    "tissue": vis_raw["tissue"],
    "delta_emax": pd.to_numeric(vis_raw["SYNERGY_DELTA_EMAX"], errors="coerce"),
})

vis = vis[
    vis["tissue"].astype(str).str.contains("pancre", case=False, na=False)
].copy()

vis["drug_a_canon"] = vis["drug_a"].map(canon_drug)
vis["drug_b_canon"] = vis["drug_b"].map(canon_drug)
vis["pair_key"] = [canon_pair(a, b) for a, b in zip(vis["drug_a"], vis["drug_b"])]

# The completed thesis workflow did not use a directly resolved Vis binary class.
# Vis therefore contributes a continuous ΔEmax/ΔEfficacy endpoint only.
vis["external_binary"] = np.nan
vis["external_continuous"] = vis["delta_emax"]
VIS_CONT = "Vis ΔEmax / ΔEfficacy"
VIS_BIN = "not available (continuous Vis endpoint only)"

print(
    "Vis PDAC rows/pairs/cells:",
    len(vis),
    vis["pair_key"].nunique(),
    vis["cell_line"].nunique(dropna=True),
)
print("Vis continuous endpoint:", VIS_CONT)

expect_count(vis["pair_key"].nunique(), 48, "Vis pair count")
expect_count(vis["cell_line"].nunique(), 30, "Vis PDAC cell lines")
assert vis["external_binary"].notna().sum() == 0


Vis PDAC rows/pairs/cells: 4476 48 30
Vis continuous endpoint: Vis ΔEmax / ΔEfficacy


In [13]:
# Bashi S18 already contains the actual compound names.
# No Supplementary Table S2 / anonymous Lxx mapping is needed.

required_bashi_cols = [
    "Compound1_name",
    "Compound2_name",
    "CELL_LINE_NAME",
    "CANCER_TYPE",
    "combo_MaxE",
    "HSA_matrix",
    "Bliss_matrix",
]

missing = [c for c in required_bashi_cols if c not in bashi_raw.columns]

if missing:
    raise KeyError(
        "The selected Bashi table is not the expected S18 response table. "
        f"Missing columns: {missing}"
    )

bashi = pd.DataFrame(
    {
        "drug_a": bashi_raw["Compound1_name"],
        "drug_b": bashi_raw["Compound2_name"],
        "cell_line": bashi_raw["CELL_LINE_NAME"],
        "cancer_type": bashi_raw["CANCER_TYPE"],
        "combo_emax": pd.to_numeric(
            bashi_raw["combo_MaxE"],
            errors="coerce",
        ),
        "hsa_matrix": pd.to_numeric(
            bashi_raw["HSA_matrix"],
            errors="coerce",
        ),
        "bliss_matrix": pd.to_numeric(
            bashi_raw["Bliss_matrix"],
            errors="coerce",
        ),
    }
)

bashi = bashi[
    bashi["cancer_type"]
    .astype(str)
    .str.contains("pancre", case=False, na=False)
].copy()

bashi["drug_a_canon"] = bashi["drug_a"].map(canon_drug)
bashi["drug_b_canon"] = bashi["drug_b"].map(canon_drug)
bashi["pair_key"] = [
    canon_pair(a, b)
    for a, b in zip(bashi["drug_a"], bashi["drug_b"])
]

#Derived Bashi combination-activity criterion.
#It is not a pre-existing binary column in the S18 workbook.

print("Bashi missing combo_MaxE:", int(bashi["combo_emax"].isna().sum()))
print("Bashi missing HSA_matrix:", int(bashi["hsa_matrix"].isna().sum()))
if bashi[["combo_emax", "hsa_matrix"]].isna().any().any():
    raise ValueError("Bashi contains missing response values: refusing to silently label them negative.")
bashi["external_binary"] = (
    (bashi["combo_emax"] > 0.5)
    & (bashi["hsa_matrix"] > 0.1)
).astype(float)

BASHI_BIN = "combo_MaxE > 0.5 AND HSA_matrix > 0.1"

# Positive HSA matrix excess = stronger interaction.
bashi["external_continuous"] = bashi["hsa_matrix"]
BASHI_CONT = "Bashi HSA matrix excess"

print("Bashi PDAC rows:", len(bashi))
print("Bashi PDAC unique pairs:", bashi["pair_key"].nunique())
print("Bashi PDAC cell lines:", bashi["cell_line"].nunique())
print("Bashi binary prevalence:", bashi["external_binary"].mean())

display(
    bashi[
        [
            "drug_a",
            "drug_b",
            "drug_a_canon",
            "drug_b_canon",
            "pair_key",
            "combo_emax",
            "hsa_matrix",
            "external_binary",
        ]
    ]
    .drop_duplicates("pair_key")
    .head(30)
)


Bashi missing combo_MaxE: 0
Bashi missing HSA_matrix: 0
Bashi PDAC rows: 2789
Bashi PDAC unique pairs: 109
Bashi PDAC cell lines: 29
Bashi binary prevalence: 0.0570096808892076


,drug_a,drug_b,drug_a_canon,drug_b_canon,pair_key,combo_emax,hsa_matrix,external_binary
8,AZD1775,Capivasertib,ADAVOSERTIB,CAPIVASERTIB,ADAVOSERTIB||CAPIVASERTIB,0.505363,0.038672,0.0
9,AZD0156,Olaparib,AZD0156,OLAPARIB,AZD0156||OLAPARIB,0.296806,0.044358,0.0
2256,AZD6738,Olaparib,CERALASERTIB,OLAPARIB,CERALASERTIB||OLAPARIB,0.360467,-0.006089,0.0
2257,AZD1775,Olaparib,ADAVOSERTIB,OLAPARIB,ADAVOSERTIB||OLAPARIB,0.254790,-0.016760,0.0
2258,AZD1775,AZD6738,ADAVOSERTIB,CERALASERTIB,ADAVOSERTIB||CERALASERTIB,0.433142,0.049756,0.0
2259,AZD2811,Olaparib,AZD2811,OLAPARIB,AZD2811||OLAPARIB,0.149112,-0.009037,0.0
2260,AZD2811,AZD6738,AZD2811,CERALASERTIB,AZD2811||CERALASERTIB,0.426507,-0.008611,0.0
2261,AZD2811,AZD1775,AZD2811,ADAVOSERTIB,ADAVOSERTIB||AZD2811,0.347266,-0.017745,0.0
2262,AZD0156,AZD1775,AZD0156,ADAVOSERTIB,ADAVOSERTIB||AZD0156,0.376986,-0.015125,0.0
2263,AZD7648,Olaparib,AZD7648,OLAPARIB,AZD7648||OLAPARIB,0.152395,-0.006923,0.0


In [14]:
#Pair-level Vis/Bashi experimental outcome tables.
#Chemical-novelty categories are assigned during external validation.

def pair_out_final(df,label,cont_name,bin_name):
    x=(df.groupby("pair_key",as_index=False)
         .agg(n_external_rows=("pair_key","size"),
              n_external_cell_lines=("cell_line",pd.Series.nunique),
              binary_positive_fraction=("external_binary","mean"),
              pair_any_positive=("external_binary","max"),
              mean_continuous_response=("external_continuous","mean"),
              median_continuous_response=("external_continuous","median")))
    parts=x["pair_key"].str.split("||",regex=False,expand=True)
    x["drug_a_canon"]=parts[0]; x["drug_b_canon"]=parts[1]
    x["dataset"]=label; x["continuous_name"]=cont_name; x["binary_name"]=bin_name
    return x


vo = pair_out_final(vis, "Vis", VIS_CONT, VIS_BIN)
bo = pair_out_final(bashi, "Bashi", BASHI_CONT, BASHI_BIN)

assert vo["pair_key"].nunique() == 48
assert bo["pair_key"].nunique() == 109
assert vo["mean_continuous_response"].notna().all()

vis_bashi=pd.concat([vo,bo],ignore_index=True)
VB_OUT=BASE/"vis_bashi_independent_external_validation_final_strict"; VB_OUT.mkdir(parents=True,exist_ok=True)
vb_path=VB_OUT/"vis_bashi_harmonized_pair_level_test_units.csv"
vis_bashi.to_csv(vb_path,index=False)
print("Saved Vis/Bashi harmonized pair-level table:",vb_path,vis_bashi.shape)


# Repository-root output guardrail.
assert PROJECT_ROOT.name.lower() != "notebooks"
assert str(vb_path).startswith(str(PROJECT_ROOT / "results")), vb_path


Saved Vis/Bashi harmonized pair-level table: C:\Users\Käyttäjä\Documents\PDAC-drug-combination-ML-MSc-thesis\results\tables\ml_comprehensive\vis_bashi_independent_external_validation_final_strict\vis_bashi_harmonized_pair_level_test_units.csv (157, 12)


## D. Source contract and reproducibility manifest

In [15]:
# Canonical output contract. All raw sources were validated before reaching this cell.
contract = pd.DataFrame([
    ["Yale", 820, False, True, yale_pair_path.relative_to(PROJECT_ROOT).as_posix()],
    ["Höfer", 146, True, True, hoefer_pair_path.relative_to(PROJECT_ROOT).as_posix()],
    ["Vis", 48, False, True, vb_path.relative_to(PROJECT_ROOT).as_posix()],
    ["Bashi", 109, True, True, vb_path.relative_to(PROJECT_ROOT).as_posix()],
], columns=["dataset", "expected_pairs", "binary_expected", "continuous_expected", "output_path"])
OUT.mkdir(parents=True, exist_ok=True)
contract_path = OUT / "external_source_harmonisation_contract.csv"
contract.to_csv(contract_path, index=False)

# Source versions and exact in-file selections; record hashes, not bundled raw data.
source_specs = [
    ("Yale", "10.1158/1535-7163.MCT-16-0794", YALE_ZIP,
     "Raw_YALE-PDAC.zip :: " + summaries[0], "observed_bliss_diff", "continuous"),
    ("Höfer", "10.1038/s44320-025-00085-6", EV2_PATH,
     HOEFER_SHEET, "Synergy; AUC∆_sum [%]", "binary and continuous"),
    ("Vis", "10.1016/j.xcrm.2024.101687", VIS_PATH,
     "Data S1 :: data-code/SupplementalMaterials/pancan_combi_fitted_data_jan21.tsv.gz",
     "SYNERGY_DELTA_EMAX", "continuous"),
    ("Bashi", "10.1158/2159-8290.CD-23-0388", BASHI_PATH,
     BASHI_SHEET, "combo_MaxE; HSA_matrix; Bliss_matrix", "derived binary and continuous"),
]
manifest = pd.DataFrame([
    {
        "study": study, "paper_doi": doi,
        "source_file": path.name, "source_sha256": sha256_file(path),
        "source_selection": selection, "source_response_columns": cols,
        "outcome_type": outcome,
    }
    for study, doi, path, selection, cols, outcome in source_specs
])
manifest_path = OUT / "external_harmonisation_source_manifest.csv"
manifest.to_csv(manifest_path, index=False)
summary_path = OUT / "external_harmonisation_run_metadata.json"
summary_path.write_text(json.dumps({
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "expected_counts_strict": ENFORCE_THESIS_PAIR_COUNTS,
    "input_pair_count_without_ncats": 820 + 146 + 48 + 109,
    "ncats_source": "PANC-1 Training_set.csv processed by external_experimental_validation.ipynb",
    "sources": {r["study"]: r["source_sha256"] for r in manifest.to_dict("records")},
}, indent=2, ensure_ascii=False), encoding="utf-8")

display(contract)
display(manifest[["study", "paper_doi", "source_selection", "source_sha256"]])
print("Saved contract:", contract_path)
print("Saved provenance:", manifest_path)
print("Saved run metadata:", summary_path)

,dataset,expected_pairs,binary_expected,continuous_expected,output_path
0,Yale,820,False,True,results/tables/ml_comprehensive/external_valid...
1,Höfer,146,True,True,results/tables/ml_comprehensive/external_hoefe...
2,Vis,48,False,True,results/tables/ml_comprehensive/vis_bashi_inde...
3,Bashi,109,True,True,results/tables/ml_comprehensive/vis_bashi_inde...


,study,paper_doi,source_selection,source_sha256
0,Yale,10.1158/1535-7163.MCT-16-0794,Raw_YALE-PDAC.zip :: Pancreas_study_summary.csv,243972ca8f62a59f91da028b59ed9dbabec898d6fe8c25...
1,Höfer,10.1038/s44320-025-00085-6,Dataset EV2_146Drugs_Synergy,7f86969a060954e4385cf6f1e815c194a2bdc70dd19fab...
2,Vis,10.1016/j.xcrm.2024.101687,Data S1 :: data-code/SupplementalMaterials/pan...,96db7c4b4e07e105fca33ea7ea2f9c6fde1daf13663fc1...
3,Bashi,10.1158/2159-8290.CD-23-0388,Supplementary Table 18 Single,6270ee50a5ac6123a7dab52aabd26954dcd513991292b1...


Saved contract: C:\Users\Käyttäjä\Documents\PDAC-drug-combination-ML-MSc-thesis\results\tables\ml_comprehensive\external_dataset_harmonisation\external_source_harmonisation_contract.csv
Saved provenance: C:\Users\Käyttäjä\Documents\PDAC-drug-combination-ML-MSc-thesis\results\tables\ml_comprehensive\external_dataset_harmonisation\external_harmonisation_source_manifest.csv
Saved run metadata: C:\Users\Käyttäjä\Documents\PDAC-drug-combination-ML-MSc-thesis\results\tables\ml_comprehensive\external_dataset_harmonisation\external_harmonisation_run_metadata.json
